In [1]:
import io
import requests
import pandas as pd
import yfinance as yf
import numpy as np

In [2]:
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

# Fetch the HTML content with headers
response = requests.get(url, headers=headers)

In [3]:
help(type(response))

Help on class Response in module requests.models:

class Response(builtins.object)
 |  The :class:`Response <Response>` object, which contains a
 |  server's response to an HTTP request.
 |
 |  Methods defined here:
 |
 |  __bool__(self)
 |      Returns True if :attr:`status_code` is less than 400.
 |
 |      This attribute checks if the status code of the response is between
 |      400 and 600 to see if there was a client error or a server error. If
 |      the status code, is between 200 and 400, this will return True. This
 |      is **not** a check to see if the response code is ``200 OK``.
 |
 |  __enter__(self)
 |
 |  __exit__(self, *args)
 |
 |  __getstate__(self)
 |      Helper for pickle.
 |
 |  __init__(self)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  __iter__(self)
 |      Allows you to use a response as an iterator.
 |
 |  __nonzero__(self)
 |      Returns True if :attr:`status_code` is less than 400.
 |
 |      This attribute checks if t

In [4]:
dfs = pd.read_html(io.StringIO(response.text))
len(dfs)

2

In [5]:
dfs[0].head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [6]:
dfs[1].head()

,vteS&P 500 companies,vteS&P 500 companies.1
0,Energy,APA Corporation Baker Hughes Chevron Corporati...
1,Materials,Air Products Albemarle Corporation Amcor Avery...
2,Industrials,3M A. O. Smith Allegion Ametek Automatic Data ...
3,Consumer discretionary,Airbnb Amazon Aptiv AutoZone Best Buy Booking ...
4,Consumer staples,Altria Archer Daniels Midland Brown-Forman Bun...


In [7]:
df = dfs[0]

In [8]:
df.dtypes

Symbol                     str
Security                   str
GICS Sector                str
GICS Sub-Industry          str
Headquarters Location      str
Date added                 str
CIK                      int64
Founded                    str
dtype: object

In [9]:
df["Date added"] = pd.to_datetime(df["Date added"], errors="coerce")
df.dtypes

Symbol                              str
Security                            str
GICS Sector                         str
GICS Sub-Industry                   str
Headquarters Location               str
Date added               datetime64[us]
CIK                               int64
Founded                             str
dtype: object

In [10]:
df[(df["Date added"].dt.year >= 2020) & (df["Date added"] <= "2026-08-21")]["Date added"].dt.year.value_counts().sort_index()

Date added
2020    10
2021    10
2022    15
2023    15
2024    16
2025    18
2026    13
Name: count, dtype: int64

In [11]:
start_date='2026-01-01'
end_date='2026-08-21'
ticker_obj = yf.Ticker("^GSPC")
spx_index = ticker_obj.history(start=start_date, end=end_date)
spx_index.tail()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-08-14 00:00:00-04:00,7806.600098,7810.009766,7776.310059,7785.759766,4159900000,0.0,0.0
2026-08-17 00:00:00-04:00,7790.680176,7790.680176,7744.879883,7745.060059,4452630000,0.0,0.0
2026-08-18 00:00:00-04:00,7700.040039,7713.950195,7688.629883,7691.759766,4602700000,0.0,0.0
2026-08-19 00:00:00-04:00,7716.740234,7743.930176,7700.069824,7707.979980,5184100000,0.0,0.0
2026-08-20 00:00:00-04:00,7690.490234,7699.959961,7639.009766,7641.160156,4759610000,0.0,0.0


In [12]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

response = requests.get(url, headers=headers)
tables = pd.read_html(io.StringIO(response.text))

# Main table is usually the first table
df = tables[0]

# Clean up columns: Ticker, Security, Date added
df["Date added"] = pd.to_datetime(df["Date added"], errors="coerce")
df["Year"] = df["Date added"].dt.year

# Filter for full years starting from 2020 (excluding ongoing incomplete year if applicable)
additions_by_year = (
    df[df["Year"] >= 2020]["Year"]
    .value_counts()
    .sort_index()
)

print("S&P 500 Additions per year:")
print(additions_by_year)

# Additional: Stocks in index for > 20 years
current_year = 2026
older_than_20 = df[df["Year"] <= (current_year - 20)]
print(f"\nCurrent S&P 500 stocks added > 20 years ago: {len(older_than_20)}")

S&P 500 Additions per year:
Year
2020    10
2021    10
2022    15
2023    15
2024    16
2025    18
2026    13
Name: count, dtype: int64

Current S&P 500 stocks added > 20 years ago: 227


In [13]:
tickers = {
    'US': '^GSPC',
    'China': '000001.SS',
    'Hong Kong': '^HSI',
    'Australia': '^AXJO',
    'India': '^NSEI',
    'Canada': '^GSPTSE',
    'Germany': '^GDAXI',
    'UK': '^FTSE',
    'Japan': '^N225',
    'Mexico': '^MXX',
    'Brazil': '^BVSP'
}

start_date = '2026-01-01'
end_date = '2026-08-21'

data = yf.download(list(tickers.values()), start=start_date, end=end_date)['Close']

# Calculate YTD return: (Last Price - First Price) / First Price
ytd_returns = {}
for country, ticker in tickers.items():
    series = data[ticker].dropna()
    if not series.empty:
        ret = (series.iloc[-1] - series.iloc[0]) / series.iloc[0] * 100
        ytd_returns[country] = ret

df_returns = pd.Series(ytd_returns)
us_return = df_returns['US']

better_than_us = df_returns[df_returns > us_return]

print(f"US (S&P 500) YTD Return: {us_return:.2f}%")
print(f"Number of indexes performing better than US: {len(better_than_us)}")
print("\nAll YTD Returns (%):")
print(df_returns.sort_values(ascending=False))

[*********************100%***********************]  11 of 11 completed


US (S&P 500) YTD Return: 11.41%
Number of indexes performing better than US: 2

All YTD Returns (%):
Japan        27.750745
Canada       14.057466
US           11.412019
UK            8.010176
Germany       5.883203
Brazil        4.601997
Australia     4.078920
Mexico        0.324972
Hong Kong    -2.429832
China        -2.974985
India        -7.322959
dtype: float64


In [15]:
# 1. Download S&P 500 data from 1950
sp500_data = yf.download('^GSPC', start='1950-01-01', end='2026-08-21')
sp500_close = sp500_data['Close'].squeeze()
sp500 = pd.DataFrame({'Close': sp500_close})

# 2. Identify All-Time Highs (ATH)
sp500['CumMax'] = sp500['Close'].cummax()
sp500['Is_ATH'] = sp500['Close'] == sp500['CumMax']

ath_dates = sp500[sp500['Is_ATH']].index

corrections = []

# 3. Analyze periods between consecutive ATHs
for i in range(len(ath_dates) - 1):
    start_date = ath_dates[i]
    end_date = ath_dates[i+1]
    
    period = sp500.loc[start_date:end_date]
    peak_price = period['Close'].iloc[0]
    min_price = period['Close'].min()
    
    drawdown = (peak_price - min_price) / peak_price * 100
    
    # 5. Filter for >= 5% drawdown
    if drawdown >= 5.0:
        duration = (end_date - start_date).days
        corrections.append({
            'Start': start_date,
            'End': end_date,
            'Drawdown_%': drawdown,
            'Duration_Days': duration
        })

df_corr = pd.DataFrame(corrections)

# 7. Calculate Percentiles
print("Drawdown (%) Percentiles:")
print(df_corr['Drawdown_%'].quantile([0.25, 0.50, 0.75]))

print("\nDuration (Days) Percentiles:")
print(df_corr['Duration_Days'].quantile([0.25, 0.50, 0.75]))

[*********************100%***********************]  1 of 1 completed


Drawdown (%) Percentiles:
0.25     6.234677
0.50     7.986358
0.75    14.019826
Name: Drawdown_%, dtype: float64

Duration (Days) Percentiles:
0.25     55.5
0.50     92.5
0.75    211.5
Name: Duration_Days, dtype: float64


In [20]:
ticker = 'AMZN'
ticker_obj = yf.Ticker(ticker)

# 1. Get earnings dates
earnings = ticker_obj.get_earnings_dates()
earnings = earnings.dropna(subset=['Reported EPS', 'Surprise(%)'])

# 2. Download historical prices safely
price_data = yf.download(ticker, start='2020-01-01', end='2026-08-21')
prices = pd.DataFrame({'Close': price_data['Close'].squeeze()})

# 3. Calculate 3-day 2-period return: Close_Day3 / Close_Day1 - 1
prices['2Day_Return'] = prices['Close'].pct_change(periods=2).shift(-1)

# Match earnings dates to returns
results = []
for date, row in earnings.iterrows():
    earnings_date = pd.to_datetime(date).tz_localize(None)
    # Find closest trading day
    idx = prices.index.get_indexer([earnings_date], method='nearest')[0]
    
    ret = prices.iloc[idx]['2Day_Return']
    surprise = row['Surprise(%)']
    
    results.append({
        'Date': prices.index[idx],
        'Surprise_%': surprise,
        'Return_2D': ret
    })

df_res = pd.DataFrame(results)

# 4. Filter for positive surprises
pos_surprises = df_res[df_res['Surprise_%'] > 0]

median_return = pos_surprises['Return_2D'].median() * 100
correlation = df_res['Surprise_%'].corr(df_res['Return_2D'])

print(f"Median 2-day Return after Positive Surprise: {median_return:.2f}%")
print(f"Correlation between Surprise Magnitude and 2-Day Return: {correlation:.4f}")

[*********************100%***********************]  1 of 1 completed

Median 2-day Return after Positive Surprise: -1.74%
Correlation between Surprise Magnitude and 2-Day Return: 0.4029


In [18]:
# Retrieving supplementary context metrics: VIX (Volatility) & Sector ETF (XLK)
extra_metrics = yf.download(['^VIX', 'XLK'], start='2024-01-01', end='2026-08-21')['Close']
print(extra_metrics.head())

[*********************100%***********************]  2 of 2 completed

Ticker            XLK   ^VIX
Date                        
2024-01-02  92.280487  13.20
2024-01-03  91.340141  14.04
2024-01-04  90.670593  14.13
2024-01-05  90.645981  13.35
2024-01-08  92.920502  13.08


In [22]:
# Define quantum computing tickers (pure-plays, key industry players, and sector ETF)
quantum_tickers = ['IONQ', 'RGTI', 'QBTS', 'IBM', 'QTUM']

# Download daily adjusted close prices for the past 2 years
quantum_data = yf.download(quantum_tickers, period='2y')['Close']

# Compute daily returns
quantum_returns = quantum_data.pct_change().dropna()

print("Quantum Stocks Close Prices:")
print(quantum_data.tail())

print("\nDaily Returns Summary:")
print(quantum_returns.describe())

[*********************100%***********************]  5 of 5 completed

Quantum Stocks Close Prices:
Ticker             IBM       IONQ       QBTS        QTUM   RGTI
Date                                                           
2026-09-08  232.089996  40.470001  17.670000  148.910004  15.81
2026-09-09  239.940002  38.139999  17.120001  147.809998  15.24
2026-09-10  234.020004  36.840000  16.660000  145.610001  15.16
2026-09-11  243.289993  36.750000  16.799999  147.410004  15.27
2026-09-14  249.089996  37.500000  16.830000  142.800003  15.27

Daily Returns Summary:
Ticker         IBM        IONQ        QBTS        QTUM        RGTI
count   499.000000  499.000000  499.000000  499.000000  499.000000
mean      0.000715    0.005605    0.009609    0.001966    0.010090
std       0.025345    0.071007    0.091447    0.020159    0.094733
min      -0.252076   -0.389998   -0.361257   -0.082347   -0.454051
25%      -0.009652   -0.035645   -0.043316   -0.008936   -0.038428
50%       0.001430   -0.002184   -0.002845    0.002818   -0.001415
75%       0.012489    0.039817

In [24]:
# 1. Fetch additional market & sector metrics alongside target stocks
tickers = {
    'IonQ': 'IONQ',
    'Rigetti': 'RGTI',
    'D-Wave': 'QBTS',
    'VIX_Index': '^VIX',
    'Semiconductor_ETF': 'SOXX'
}

start_date = '2024-01-01'
end_date = '2026-08-21'

# Download daily close data
data = yf.download(list(tickers.values()), start=start_date, end=end_date)
close_prices = data['Close'].rename(columns={v: k for k, v in tickers.items()})

# 2. Compute 14-day rolling volatility and correlation with VIX
returns = close_prices.pct_change()
volatility_14d = returns.rolling(window=14).std()

# Correlation matrix between quantum stocks and external indicators
corr_matrix = returns.corr()

print("Latest Data Sample:")
print(close_prices.tail())

print("\nReturn Correlation Matrix:")
print(corr_matrix[['VIX_Index', 'Semiconductor_ETF']])

[*********************100%***********************]  5 of 5 completed

Latest Data Sample:
Ticker           IonQ     D-Wave    Rigetti  Semiconductor_ETF  VIX_Index
Date                                                                     
2026-08-14  46.259998  21.170000  18.820000         550.419983      14.25
2026-08-17  46.840000  20.870001  18.670000         559.119995      15.19
2026-08-18  44.119999  19.530001  17.709999         531.390015      15.84
2026-08-19  43.360001  19.320000  17.000000         519.669983      14.89
2026-08-20  41.529999  18.799999  16.065001         522.349976      16.01

Return Correlation Matrix:
Ticker             VIX_Index  Semiconductor_ETF
Ticker                                         
IonQ               -0.372519           0.412056
D-Wave             -0.245834           0.310363
Rigetti            -0.288370           0.341130
Semiconductor_ETF  -0.614399           1.000000
VIX_Index           1.000000          -0.614399
